# 4.32 抓取 2024 年北京高考数据

In [4]:
import os
import pandas as pd
from p3lib.ch3 import url_to_filename, PageDownloader

In [2]:
data_dir = 'beijing'
crawler = PageDownloader(save_dir=data_dir)
#抓取北京市2024年本科普通批录取投档线
target_url = "https://gaokao.eol.cn/bei_jing/dongtai/202407/t20240720_2625168.shtml"
crawler.process(target_url)
score_file = os.path.join(data_dir, f'{url_to_filename(target_url)}.html')
assert(os.path.exists(score_file)) #检查目标网页是否已保存到指定路径
#抓取2024年北京高考分段表
target_url = "https://gaokao.eol.cn/bei_jing/dongtai/202406/t20240625_2619192.shtml"
crawler.process(target_url)
rank_file = os.path.join(data_dir, f'{url_to_filename(target_url)}.html')
assert(os.path.exists(rank_file))

[Status] 开始抓取 https://gaokao.eol.cn/bei_jing/dongtai/202407/t20240720_2625168.shtml
[Status] 已保存到 beijing/aHR0cHM6Ly9nYW9rYW8uZW9sLmNuL2JlaV9qaW5nL2Rvbmd0YWkvMjAyNDA3L3QyMDI0MDcyMF8yNjI1MTY4LnNodG1s.html
[Status] 页面抓取成功
[Status] 开始抓取 https://gaokao.eol.cn/bei_jing/dongtai/202406/t20240625_2619192.shtml
[Status] 已保存到 beijing/aHR0cHM6Ly9nYW9rYW8uZW9sLmNuL2JlaV9qaW5nL2Rvbmd0YWkvMjAyNDA2L3QyMDI0MDYyNV8yNjE5MTkyLnNodG1s.html
[Status] 页面抓取成功


# 4.33 投档线数据清洗

In [6]:
score_df = pd.read_html(score_file)[0]
score_df = score_df[1:].reset_index(drop=True) #跳过原表第一行，并重置行索引
score_df.columns = ['院校代码', '院校', '专业组代码', '限制', '投档线'] #列重命名
score_df['投档线'] = score_df['投档线'].astype(int) 
score_df['投档线']

0       700
1       698
2       690
3       689
4       688
       ... 
1253    434
1254    434
1255    434
1256    434
1257    434
Name: 投档线, Length: 1258, dtype: int64

# 4.34 统计本科普通批次各高校专业组数量

In [7]:
univ_df = score_df['院校'].value_counts().to_frame().reset_index()
univ_df.columns = ['院校', '专业组数量']
print(f'普通批次共有{len(univ_df)}家高校、{len(score_df)}个专业组') # 普通批次共有591家高校、1258个专业组
major_group = score_df['限制'].value_counts().to_frame().reset_index() #统计专业填报限制情况

普通批次共有591家高校、1258个专业组


# 4.35 分段表清洗并规整

In [9]:
rank_df = pd.read_html(rank_file)[0]
rank_df = rank_df[1:].reset_index(drop=True)
rank_df.columns = ['分数', '人数', '位次']
rank_df.iloc[0,0]= "700" #将第一行的700-750替换为700
hit = rank_df['分数'].str.contains(r'本科线', na=False) #正则匹配，查找含有'本科线'字样的行
cutoff = hit.idxmax() #定位本科线对应的行
rank_df = rank_df[:cutoff+1] # 舍弃该行之后的数据
rank_df['分数'] = rank_df['分数'].str.replace(r'（.*?）', '', regex=True) #清除注释（特招线、本科线）
rank_df = rank_df.astype(int)
rank_df

,分数,人数,位次
0,700,117,117
1,699,15,132
2,698,16,148
3,697,17,165
4,696,18,183
...,...,...,...
262,438,170,44347
263,437,156,44503
264,436,149,44652
265,435,137,44789


# 4.36 分数转位次

In [11]:
score_rank = rank_df.set_index('分数')['位次'] # 先构造“分数->位次”的映射表
score_df['位次'] = score_df['投档线'].map(score_rank)
score_df

,院校代码,院校,专业组代码,限制,投档线,位次
0,1021,北京大学,2,不限,700,117
1,1023,清华大学,2,物理,698,148
2,1021,北京大学,3,物理＋化学,690,354
3,1023,清华大学,3,物理＋化学,689,384
4,1021,北京大学,1,不限,688,423
...,...,...,...,...,...,...
1253,6518,喀什大学,3,物理＋化学,434,44923
1254,6519,新疆政法学院,2,物理＋化学,434,44923
1255,6521,新疆和田学院,3,物理＋化学,434,44923
1256,6522,昌吉学院,3,物理＋化学,434,44923


# 4.37 查找指定位次区间的专业组

In [14]:
n_24, n_25 = 6.72, 7.89 # 2024, 2025年北京高考人数（单位：万）
rank_25 = 16470 # 考生当前位次
query = int(rank_25*n_24/n_25+0.5) # 按人数比值修正位次
error = 0.05                      
low  = int(query * (1 - error))
high = int(query * (1 + error))
hit = score_df[(score_df['位次'] >= low) & (score_df['位次'] <= high)]
print(f'2025年位次: {rank_25}，对应2024年位次: {query}')
print(f'位次在[{low}, {high}]的专业组数：{len(hit)}')

2025年位次: 16470，对应2024年位次: 14028
位次在[13326, 14729]的专业组数：47


In [15]:
sub = hit[hit['限制'].str.contains(r'不限') == True]
print(f'无限制的专业组数：{len(sub)}')
sub = hit[hit['限制'].str.contains(r'物理$') == True] 
print(f'限"物理"的专业组数：{len(sub)}')

无限制的专业组数：10
限"物理"的专业组数：7
